# Generating Synthetic Gaze with the Geometrical Generator

The {py:class}`GeometricalSampleGenerator <pymovements.synthetic.GeometricalSampleGenerator>`
generates gaze positions from a sequence of fixations. Every sample inside a fixation takes the
fixation's location, and every other sample is interpolated between the locations around it. The
two interpolation methods are the ones recommended for filling gaps in recorded gaze by:

> Grootjen, J. W., Weingärtner, H., & Mayer, S. (2024).
> *Uncovering and Addressing Blink-Related Challenges in Using Eye Tracking for Interactive
> Systems.* CHI '24.

Using the fixation centres as the positions the gaze moves between follows the mapping step in
section 3.7 of:

> Fuhl, W., & Kasneci, E. (2018). *Eye movement velocity and gaze data generator for evaluation,
> robustness testing and assess of eye tracking software and visualization tools.*
> arXiv:1808.09296.

The events are therefore the ground truth of the generated samples. The generator emits
**positions**, so its output can go through the usual preprocessing and event detection. The
movement between two fixations is interpolated, not modelled: its duration is the gap the events
leave, and it has no velocity profile of its own.

## What you will learn in this tutorial:

* how to generate samples from fixations with a location
* how to plot the generated positions
* how to choose between linear and cubic spline interpolation
* how to label the gaps between fixations as saccades
* how to add scatter to the fixations
* how to generate positions in pixels
* how to generate several trials
* how to reproduce a run from its metadata
* how to generate samples for fixations detected in a real recording

## Preparations

We import `pymovements` as the alias `pm` for convenience, `polars` to write the events, and
`numpy` and `matplotlib` to compare signals.

The generation interface is still **experimental**: names and behaviour may change without that
counting as a breaking change, so pin the pymovements version if you depend on it. Each run warns
about this, which we silence here for readability.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

import pymovements as pm

warnings.filterwarnings('ignore', category=pm.ExperimentalWarning)

## Generating Samples from Fixations

The generator needs fixations with a `location`, the position the gaze holds during the fixation.
Here we write three of them by hand, in degrees of visual angle: a jump of 10 degrees along the x
axis over a gap of 50 ms, and a jump of 5 degrees along the y axis over a gap of 60 ms.

In [ ]:
events = pm.Events(
    pl.DataFrame({
        'name': ['fixation', 'fixation', 'fixation'],
        'onset': [0, 300, 620],
        'offset': [250, 560, 900],
        'location': [[0.0, 0.0], [10.0, 0.0], [10.0, 5.0]],
    }),
    time_unit='ms',
)

events.frame

The constructor of the generator takes the sampling rate and the hyperparameters, which all have
defaults. The events and the seed are passed to
{py:meth}`generate() <pymovements.synthetic.GeometricalSampleGenerator.generate>`, as they belong
to the run and not to the generator.

In [ ]:
generator = pm.synthetic.GeometricalSampleGenerator(sampling_rate=1000)

gaze = generator.generate(events=events, seed=42)

gaze.samples

The output is a {py:class}`Gaze <pymovements.Gaze>` with one sample per millisecond. It carries
the events it was generated from, so the ground truth stays with the samples:

In [ ]:
gaze.events.frame

## Plotting the Positions

{py:func}`tsplot() <pymovements.plotting.tsplot>` draws each component of the positions against
the `time` column, and `show_events=True` shades the events. The flat stretches are the
fixations, and the gaze moves between them in the unshaded gaps.

In [ ]:
pm.plotting.tsplot(
    gaze,
    'position',
    show_events=True,
    title='geometrical generator, 1000 Hz',
)

As the output has coordinates, {py:func}`traceplot() <pymovements.plotting.traceplot>` applies as
well. It draws the path of the gaze, here from the first location to the third, with the y axis
pointing downwards as on a screen:

In [ ]:
pm.plotting.traceplot(gaze, position_column='position', title='path of the gaze')

## Choosing the Interpolation

`interpolation` sets how the gaze moves between two fixations. `'linear'` gives a constant velocity
in each gap, and the default `'cubic_spline'` gives a smooth movement whose velocity peaks around
the middle of the gap. We compare them through the velocity that
{py:meth}`pos2vel() <pymovements.Gaze.pos2vel>` computes:

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 6), sharex=True)

for ax, interpolation in zip(axes, ('linear', 'cubic_spline')):
    moved = pm.synthetic.GeometricalSampleGenerator(
        sampling_rate=1000,
        interpolation=interpolation,
    ).generate(events=events, seed=42)
    moved.pos2vel()

    time = moved.samples['time'].dt.total_milliseconds().to_numpy()
    velocity = np.array(moved.samples['velocity'].to_list(), dtype=float)
    speed = np.linalg.norm(velocity, axis=1)

    ax.plot(time, speed)
    ax.set_title(f'{interpolation}: peak of {np.nanmax(speed):.0f} deg/s')
    ax.set_ylabel('speed [deg/s]')

axes[1].set_xlabel('time [ms]')
plt.show()

Neither shape is a measured saccade, and the peak follows from the length of the jump and of the
gap, not from a main sequence. Both shapes are also symmetric, which recorded saccades usually are
not, especially large ones.

The cubic spline is fitted over all fixations of a trial, not per gap, so it can overshoot a
location at low rates or around very short fixations. Here a fixation of 10 ms at 100 Hz sends it
past the location of 10 degrees, while linear interpolation never leaves the range of the
locations:

In [ ]:
short_fixation = pm.Events(
    pl.DataFrame({
        'name': ['fixation', 'fixation', 'fixation'],
        'onset': [0, 300, 600],
        'offset': [100, 310, 700],
        'location': [[0.0, 0.0], [10.0, 0.0], [10.0, 0.0]],
    }),
    time_unit='ms',
)

for interpolation in ('linear', 'cubic_spline'):
    moved = pm.synthetic.GeometricalSampleGenerator(
        sampling_rate=100,
        interpolation=interpolation,
    ).generate(events=short_fixation, seed=42)

    largest = np.array(moved.samples['position'].to_list())[:, 0].max()
    print(f'{interpolation}: largest x of {largest:.2f} degrees')

## Labelling the Gaps as Saccades

This generator can render a gap, so fixations go in as they are, and the attached events keep their
gaps. If you want the movements in the ground truth,
{py:func}`bridge_gaps() <pymovements.synthetic.bridge_gaps>` fills each gap with a saccade first.
The samples stay the same, because a saccade is interpolated like a gap. Only the attached events
change:

In [ ]:
bridged = pm.synthetic.bridge_gaps(events)

labelled = generator.generate(events=bridged, seed=42)

print('same samples:', labelled.samples.equals(gaze.samples))
print('events:', labelled.events.frame['name'].to_list())

Bridging is a separate step because not every gap is a saccade: a long gap is more likely a blink
or data loss. With `max_gap`, gaps longer than the given number of milliseconds are left untouched.

The inserted saccades have an `onset` and a `duration` but no `offset`. As
{py:func}`tsplot() <pymovements.plotting.tsplot>` currently needs an `offset` to shade an event, we
plot bridged events without `show_events=True`.

## Adding Scatter to the Fixations

By default the output is free of noise, and a fixation is exactly still. `fixation_deviation` adds
a scatter around the location, in the unit of the locations. Per fixation, a deviation is drawn
from the {py:class}`Range <pymovements.synthetic.Range>`, and used as the standard deviation of a
normal scatter per sample and axis. You can give it as a `(minimum, maximum)` pair for a uniform
range, as a single value, or as a `Range` to choose another distribution.

In [ ]:
scattered = pm.synthetic.GeometricalSampleGenerator(
    sampling_rate=1000,
    fixation_deviation=(0.1, 0.3),
).generate(events=events, seed=42)

pm.plotting.tsplot(
    scattered,
    'position',
    show_events=True,
    title='fixation_deviation of 0.1 to 0.3 degrees',
)

The scatter is added after the interpolation, so the gaps follow the locations themselves.

The values above are chosen to be visible in the plot, and they are far too large for event
detection. The scatter is drawn independently for each sample, and `pos2vel` turns it into
velocity: at 1000 Hz, a difference of 0.1 degrees between two samples is a speed of 100 deg/s.
On the five ToyDataset recordings, IVT still finds the fixations up to a deviation of 0.01 degrees,
about 78 % of them at 0.02 degrees, and almost none at 0.03 degrees, which is about the spread
of the recorded fixations. Keep `fixation_deviation` at 0.01 degrees or below if the output goes
through a velocity-based detector, and do not read it as a model of fixational eye movements.

Such a model would correlate neighbouring samples. For example, section 4.1 of the following
work simulates microsaccades by passing Gaussian white noise through a 1/f^α filter:

> Lan, G., Scargill, T., & Gorlatova, M. (2022). *EyeSyn: Psychology-inspired Eye Movement
> Synthesis for Gaze-based Activity Recognition.* IPSN '22.

## Generating Positions in Pixels

The generator does not convert units, so `location_column_group` has to match the unit of the
locations. With `'pixel'`, the positions are written to the `pixel` column. To go on to degrees,
{py:meth}`pix2deg() <pymovements.Gaze.pix2deg>` needs the geometry of the screen, which we set on
the output, as the generator only knows the sampling rate:

In [ ]:
pixel_events = pm.Events(
    pl.DataFrame({
        'name': ['fixation', 'fixation'],
        'onset': [0, 300],
        'offset': [250, 560],
        'location': [[512.0, 384.0], [800.0, 384.0]],
    }),
    time_unit='ms',
)

in_pixels = pm.synthetic.GeometricalSampleGenerator(
    sampling_rate=1000,
    location_column_group='pixel',
).generate(events=pixel_events, seed=42)

in_pixels.experiment = pm.Experiment(
    screen_width_px=1024,
    screen_height_px=768,
    screen_width_cm=38,
    screen_height_cm=30,
    distance_cm=68,
    origin='upper left',
    sampling_rate=1000,
)
in_pixels.pix2deg()

in_pixels.samples.head(3)

## Generating Several Trials

Events with trial columns are generated trial by trial, each as its own continuous signal, so the
time between trials is never filled. The trial columns appear in front of the samples:

In [ ]:
trial_events = pm.Events(
    pl.DataFrame({
        'trial': [1, 1, 2, 2],
        'name': ['fixation'] * 4,
        'onset': [0, 300, 0, 200],
        'offset': [250, 560, 150, 400],
        'location': [[0.0, 0.0], [10.0, 0.0], [5.0, 5.0], [-5.0, 5.0]],
    }),
    trial_columns='trial',
    time_unit='ms',
)

trials = generator.generate(events=trial_events, seed=42)

trials.samples.group_by('trial', maintain_order=True).len()

## Reproducing a Run

Each run records the generator, its hyperparameters and the seed in the metadata of its output,
including when the seed was drawn for you.

In [ ]:
unseeded = pm.synthetic.GeometricalSampleGenerator(
    sampling_rate=1000,
    fixation_deviation=pm.synthetic.Range(0.1, 0.3, 'normal', std=0.05),
).generate(events=events)

record = unseeded.metadata['synthetic']

print('generator:', record['generator'])
print('drawn seed:', record['seed'])
print('deviation:', record['hyperparameters']['fixation_deviation'])

The generator is registered under its name, so
{py:func}`generate_samples() <pymovements.synthetic.generate_samples>` can rebuild it from the
record. Together with the same events, this reproduces the signal exactly:

In [ ]:
repeated = pm.synthetic.generate_samples(
    record['generator'],
    events=events,
    seed=record['seed'],
    **record['hyperparameters'],
)

print('reproduced:', repeated.samples.equals(unseeded.samples))

## Generating Samples for a Real Recording

The events do not have to be handmade. Detect fixations in a recording and compute their locations,
and the generator follows them: real event timing and positions, synthetic samples. We use the
{ref}`ToyDataset`, and {py:meth}`compute_event_properties() <pymovements.Gaze.compute_event_properties>`
gives each fixation its `location`:

In [ ]:
dataset = pm.Dataset('ToyDataset', path='data/ToyDataset')
dataset.download()
dataset.load()

recording = dataset.gaze[0]
recording.pix2deg()
recording.pos2vel()
recording.detect('ivt')
recording.compute_event_properties('location')

located = pm.Events(recording.events.frame.select('name', 'onset', 'offset', 'location'))

located.frame.head(3)

The locations are in degrees, like the `position` column of the recording, so the default
`location_column_group` fits. We use the sampling rate of the recording:

In [ ]:
from_recording = pm.synthetic.generate_samples(
    'geometrical',
    events=located,
    sampling_rate=recording.experiment.sampling_rate,
    seed=42,
)

start = located.frame['onset'][0]


def first_seconds(samples, seconds=3):
    """Return the time since the first fixation in ms, and x for the first seconds."""
    time = (samples['time'] - start).dt.total_milliseconds().to_numpy()
    x = np.array(samples['position'].to_list(), dtype=float)[:, 0]
    inside = (time >= 0) & (time <= seconds * 1000)
    return time[inside], x[inside]


plt.figure(figsize=(15, 4))
plt.plot(*first_seconds(recording.samples), label='recording', alpha=0.6)
plt.plot(*first_seconds(from_recording.samples), label='geometrical generator')
plt.xlabel('time since the first fixation [ms]')
plt.ylabel('x [deg]')
plt.legend()
plt.show()

### Detecting the Fixations Again

As the output has positions, the detector that found the fixations can run on it. If the generator
kept what the detector needs, the same fixations come back:

In [ ]:
from_recording.pos2vel()
from_recording.detect('ivt')

detected_again = from_recording.events.frame.slice(located.frame.height)

print('fixations in the recording:', located.frame.height)
print('fixations detected again:', detected_again.height)

onset_shift = (detected_again['onset'] - located.frame['onset']).dt.total_milliseconds()
offset_shift = (detected_again['offset'] - located.frame['offset']).dt.total_milliseconds()

pl.DataFrame({
    'shift': ['onset', 'offset'],
    'smallest [ms]': [onset_shift.min(), offset_shift.min()],
    'largest [ms]': [onset_shift.max(), offset_shift.max()],
})

All 73 fixations of this recording come back. Their onsets shift by -10 to +2 ms and their
offsets by -3 to +9 ms, as the spline leaves and approaches a fixation slowly. The speed at the
edges of a gap therefore stays below the threshold of the detector for a few milliseconds, and a
fixation tends to start earlier and end later.

This is not true of every recording. Across all five ToyDataset recordings, 534 of 551
fixations come back. The lost ones mostly follow a long gap with a small jump, so the speed
through the gap stays below the threshold of IVT and no saccade separates the two fixations.

This shows that the timing and the locations survive the round trip, not that the signal matches
the recording. The plot above shows what the generator does not model: the recorded x overshoots
after most saccades and then settles, while the generated x goes straight from one location to
the next.

## What you have learned in this tutorial:

* generating samples from fixations with a location with
  {py:class}`GeometricalSampleGenerator <pymovements.synthetic.GeometricalSampleGenerator>`
* plotting the positions with {py:func}`tsplot() <pymovements.plotting.tsplot>` and
  {py:func}`traceplot() <pymovements.plotting.traceplot>`
* choosing between linear and cubic spline interpolation
* labelling the gaps as saccades with {py:func}`bridge_gaps() <pymovements.synthetic.bridge_gaps>`
* adding scatter to the fixations with `fixation_deviation`
* generating positions in pixels and generating several trials
* reproducing a run from its metadata with
  {py:func}`generate_samples() <pymovements.synthetic.generate_samples>`
* generating samples for fixations detected in a real recording, and detecting them again